# Chapter 4 companion: difference equations and the inverse z-transform.

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Course sources: Mario Sznaier's lecture notes; T. Charalambous's slides; Phillips, Nagle and Chakrabortty, *Digital Control System Analysis and Design*, 4th ed., Pearson (2015).

Reproduces:
  * one convolution two ways: step * 0.5^k = 2 - 0.5^k;
  * m(k) = e(k) - e(k-1) - m(k-1) with a step input: m(k) = (-1)^k;
  * x(k+2) - 3x(k+1) + 2x(k) = 0, x(0) = 0, x(1) = 1: X(z) = z/((z-1)(z-2)),
    inverted three ways (partial fractions, long division, residues): 2^k - 1;
  * a repeated pole: z/((z-1)^2 (z-2)) -> 2^k - k - 1;
  * no factor z in the numerator: 1/(z - 0.5);
  * Parseval: sum (0.5^k)^2 = 4/3, in time and in frequency;
  * the exit ticket: z^2/((z-1)(z-0.2)) -> 5/4 - (1/4) 0.2^k.

Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import quad

FIG = Path.cwd() / "figures"
FIG.mkdir(exist_ok=True)
N = 10
k = np.arange(N)

In [ ]:
def long_division(num, den, n):
    """First n samples of E(z) = num/den (descending powers of z, deg num <= deg den):
    the impulse response of the filter with the same coefficients in z^-1."""
    num = np.r_[np.zeros(len(den) - len(num)), num]
    imp = np.r_[1.0, np.zeros(n - 1)]
    return signal.lfilter(num, den, imp)

In [ ]:
def residues_over_z(num, den):
    """Partial fractions of E(z)/z: returns residues r_i and poles p_i (simple poles)."""
    r, p, _ = signal.residue(num, np.polymul(den, [1, 0]))
    return r, p

In [ ]:
# ---------------------------------------------------------------- convolution
f_time = np.convolve(np.ones(N), 0.5**k)[:N]

In [ ]:
f_z = long_division([1, 0, 0], np.polymul([1, -1], [1, -0.5]), N)    # z^2/((z-1)(z-0.5))
print("Convolution of the unit step with 0.5^k")
print("  direct sum      :", np.round(f_time, 4))
print("  from E1(z)E2(z) :", np.round(f_z, 4))
print("  2 - 0.5^k       :", np.round(2 - 0.5**k, 4))

In [ ]:
# ---------------------------------------------------------------- difference equation, step input
e = np.ones(N)

In [ ]:
m = np.zeros(N)

In [ ]:
for kk in range(N):
    m[kk] = e[kk] - (e[kk - 1] if kk else 0) - (m[kk - 1] if kk else 0)

In [ ]:
print("\nm(k) = e(k) - e(k-1) - m(k-1), unit step:", m.astype(int), " (M(z) = z/(z+1): (-1)^k)")

In [ ]:
# ---------------------------------------------------------------- initial conditions, three inversions
x = np.zeros(N)

In [ ]:
x[1] = 1

In [ ]:
for kk in range(N - 2):
    x[kk + 2] = 3 * x[kk + 1] - 2 * x[kk]

In [ ]:
num, den = [1, 0], [1, -3, 2]                      # X(z) = z/((z-1)(z-2))
r, p = residues_over_z(num, den)
pf = sum(ri * pi**k for ri, pi in zip(r, p) if abs(pi) > 1e-12)
print("\nx(k+2) - 3x(k+1) + 2x(k) = 0, x(0) = 0, x(1) = 1")
print("  recursion       :", x.astype(int))
print("  partial fractions of X(z)/z: residues", np.round(r, 6), "at poles", np.round(p, 6))
print("  sum k_i p_i^k   :", np.round(pf.real).astype(int))
print("  long division   :", np.round(long_division(num, den, N)).astype(int))
res = [(2.0**kk) / (2 - 1) + 1.0 / (1 - 2) for kk in k]   # residues of z^k/((z-1)(z-2)) at 2 and 1
print("  residues        :", np.round(res).astype(int), " -> x(k) = 2^k - 1")

In [ ]:
# ---------------------------------------------------------------- repeated pole
num, den = [1, 0], np.polymul(np.polymul([1, -1], [1, -1]), [1, -2])

In [ ]:
r, p, _ = signal.residue(num, np.polymul(den, [1, 0]))
print("\nRepeated pole, E(z) = z/((z-1)^2 (z-2))")
print("  E(z)/z: residues", np.round(r, 6) + 0.0, "poles", np.round(p, 6) + 0.0)
print("  (the repeated pole is listed twice: k11 for 1/(z-1), then k12 for 1/(z-1)^2)")
print("  long division   :", np.round(long_division(num, den, N)).astype(int))
print("  2^k - k - 1     :", (2**k - k - 1))

In [ ]:
# ---------------------------------------------------------------- no factor z
print("\nE(z) = 1/(z - 0.5): long division", np.round(long_division([1], [1, -0.5], 6), 4),
      " = -2 delta(k) + 2 (0.5)^k:", np.round(-2 * (k[:6] == 0) + 2 * 0.5 ** k[:6], 4))

In [ ]:
# ---------------------------------------------------------------- Parseval
energy_time = np.sum(0.25 ** np.arange(200))

In [ ]:
energy_freq = quad(lambda th: 1 / (1.25 - np.cos(th)), 0, 2 * np.pi)[0] / (2 * np.pi)
print(f"\nParseval for 0.5^k: time {energy_time:.6f}, frequency {energy_freq:.6f}, exact 4/3 = {4 / 3:.6f}")

In [ ]:
# ---------------------------------------------------------------- exit ticket
num, den = [1, 0, 0], np.polymul([1, -1], [1, -0.2])

In [ ]:
print("\nExit ticket, E(z) = z^2/((z-1)(z-0.2))")
print("  long division      :", np.round(long_division(num, den, 6), 4))
print("  5/4 - (1/4) 0.2^k  :", np.round(1.25 - 0.25 * 0.2 ** k[:6], 4))

In [ ]:
# check yourself 1: (3z^2 + z)/((z-1)(z+0.5))
num, den = [3, 1, 0], np.polymul([1, -1], [1, 0.5])

In [ ]:
print("Check yourself, (3z^2 + z)/((z-1)(z+0.5)):", np.round(long_division(num, den, 5), 4),
      " = 8/3 + (1/3)(-0.5)^k:", np.round(8 / 3 + (-0.5) ** k[:5] / 3, 4))

In [ ]:
# ---------------------------------------------------------------- figure
th = np.linspace(0, 2 * np.pi, 400)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].stem(k, f_time)
axes[0].axhline(2, ls="--", color="gray", lw=0.8)
axes[0].set(xlabel="k", ylabel="f(k)", title="Step * 0.5^k = 2 - 0.5^k")
axes[1].plot(th, 1 / (1.25 - np.cos(th)))
axes[1].axhline(4 / 3, ls="--", color="gray", lw=0.8, label="mean = 4/3")
axes[1].set(xlabel="theta (rad)", ylabel="|F(e^{j theta})|^2", title="Parseval: same energy")
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ch04_convolution_parseval.png", dpi=150)
print("\nSaved figures/ch04_convolution_parseval.png")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Pole positions and the inverse sequence.** Take $E(z) = z/\big((z-p_1)(z-p_2)\big)$.
Partial fractions of $E(z)/z$ give $e_k = k_1 p_1^k + k_2 p_2^k$ with
$k_1 = 1/(p_1 - p_2)$, $k_2 = 1/(p_2 - p_1)$; the defaults $p_1 = 1$, $p_2 = 2$ give
$e_k = 2^k - 1$. Move the poles: each one contributes its own mode $p_i^k$, a negative pole
flips sign every sample, and one pole outside the unit circle is enough to make $e_k$ grow.
Set $p_1 = p_2$ to see the repeated-pole case, $e_k = k\,p^{k-1}$.
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
NK = 13       # k = 0..12


def explore(p1=1.0, p2=2.0):
    p1, p2 = round(p1, 10), round(p2, 10)
    k = np.arange(NK)
    # long division: the impulse response of z^-1 / (1 - (p1 + p2) z^-1 + p1 p2 z^-2)
    den = [1.0, -(p1 + p2), p1 * p2]
    e_div = signal.lfilter([0.0, 1.0, 0.0], den, np.r_[1.0, np.zeros(NK - 1)])
    repeated = abs(p1 - p2) < 1e-9
    if repeated:
        with np.errstate(divide="ignore", invalid="ignore"):
            e_pf = np.where(k == 0, 0.0, k * p1 ** np.maximum(k - 1, 0))
        modes = None
    else:
        k1, k2 = 1 / (p1 - p2), 1 / (p2 - p1)
        m1, m2 = k1 * p1 ** k, k2 * p2 ** k
        e_pf = m1 + m2
        modes = (m1, m2)

    rho = max(abs(p1), abs(p2))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.0),
                                   gridspec_kw={"width_ratios": [1, 1.9]})
    th = np.linspace(0, 2 * np.pi, 300)
    ax1.plot(np.cos(th), np.sin(th), color=GRAY, lw=1)
    ax1.axhline(0, color="k", lw=0.5)
    ax1.axvline(0, color="k", lw=0.5)
    ax1.plot([p1, p2], [0, 0], "x", color=ORANGE, ms=12, mew=2.5, label="poles")
    ax1.plot([0], [0], "o", mfc="none", color=BLUE, ms=9, mew=1.8, label="zero")
    lim = max(1.3, 1.15 * rho)
    ax1.set_xlim(-lim, lim)
    ax1.set_ylim(-lim, lim)
    ax1.set_aspect("equal")
    ax1.set_xlabel("Re z")
    ax1.set_ylabel("Im z")
    ax1.set_title(r"$E(z) = z/((z-p_1)(z-p_2))$", fontsize=11)
    ax1.legend(fontsize=10, loc="upper left")

    if modes is not None:
        ax2.plot(k, modes[0], "s--", color=TEAL, ms=4, lw=1, label=r"mode $k_1 p_1^k$")
        ax2.plot(k, modes[1], "^--", color=GRAY, ms=4, lw=1, label=r"mode $k_2 p_2^k$")
    ml, sl, bl = ax2.stem(k, e_pf, label=r"$e_k$, partial fractions")
    plt.setp(ml, color=ORANGE, markersize=6)
    plt.setp(sl, color=ORANGE, lw=2)
    plt.setp(bl, color="k", lw=0.6)
    ax2.plot(k, e_div, "x", color=BLUE, ms=9, mew=1.6, label="long division")
    ax2.set_xlabel("k")
    ax2.set_ylabel(r"$e_k$")
    verdict = "decays" if rho < 1 - 1e-12 else ("grows" if rho > 1 + 1e-12 else "does not decay")
    ax2.set_title(f"$p_1$ = {p1:g}, $p_2$ = {p2:g}: $e_k$ {verdict}", fontsize=11)
    ax2.legend(fontsize=10, loc="best")
    fig.tight_layout()

    if repeated:
        print(f"repeated pole p = {p1:g}: E(z) = z/(z-p)^2  ->  e_k = k p^(k-1)")
    else:
        print(f"k1 = 1/(p1 - p2) = {k1:.4g},  k2 = 1/(p2 - p1) = {k2:.4g}  ->  "
              f"e_k = ({k1:.4g})({p1:g})^k + ({k2:.4g})({p2:g})^k")
    print("e_0..e_6 =", np.round(e_pf[:7], 4),
          "(long division agrees)" if np.allclose(e_pf, e_div, rtol=1e-9, atol=1e-9) else "")
    print(f"largest pole magnitude {rho:.3g}: e_k {verdict}")
    plt.show()


try:
    from ipywidgets import interact, FloatSlider
    interact(explore,
             p1=FloatSlider(value=1.0, min=-1.2, max=2.0, step=0.05, description="p1",
                            continuous_update=False),
             p2=FloatSlider(value=2.0, min=-1.2, max=2.0, step=0.05, description="p2",
                            continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 4.** These cells go with the Hands-on exercises at the end of
Chapter 4. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
# ---------------------------------------------------------------- H1
def h1_two_lakes(release=0.2):
    """H1: s(k+1) = (1-r) s(k) + 100 delta(k), l(k+1) = (1-r) l(k) + r s(k)."""
    n = 30
    s, l = np.zeros(n), np.zeros(n)
    for k in range(n - 1):
        s[k + 1] = (1 - release) * s[k] + (100.0 if k == 0 else 0.0)
        l[k + 1] = (1 - release) * l[k] + release * s[k]
    k = np.arange(n)
    kp = int(np.argmax(l))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(k, s, "o-", color=BLUE, ms=4, lw=1, label="upper lake s(k)")
    ax.plot(k, l, "s-", color=ORANGE, ms=4, lw=1, label="lower lake l(k)")
    ax.set_xlabel("day k")
    ax.set_ylabel("excess volume")
    ax.set_title(f"release {release:.2f}/day: lower lake peaks on day {kp} at {l[kp]:.2f}; "
                 f"total outflow {release * l.sum():.1f} (of 100) in {n} days", fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_battery_pulse():
    """H2: pulse response of G(z) = 0.3/((z-0.5)(z-0.8)) against the classmate's 0.8^k - 0.5^k."""
    n = 16
    _, (g,) = signal.dimpulse(([0.3], [1, -1.3, 0.4], 1), n=n)
    g = g.ravel()
    k = np.arange(n)
    flawed = 0.8 ** k - 0.5 ** k
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.plot(k, g, "o-", color=ORANGE, ms=6, lw=1, label="pulse response of G(z)")
    ax.plot(k, flawed, "s--", color=TEAL, ms=4, lw=1, label="classmate: 0.8^k - 0.5^k")
    ax.set_xlabel("k")
    ax.set_ylabel("temperature rise (C)")
    ax.set_title(f"hottest at k = {int(np.argmax(g))} (classmate: k = {int(np.argmax(flawed))})",
                 fontsize=11)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_smoothing(a=0.5):
    """H3: y(k) = a y(k-1) + (1-a) x(k) on 0.5 m white noise, and on a 1 m climb."""
    rng = np.random.default_rng(0)
    n = 20000
    x = 0.5 * rng.standard_normal(n)
    y = signal.lfilter([1 - a], [1, -a], x)
    std = y[1000:].std()
    k = np.arange(120)
    climb = signal.lfilter([1 - a], [1, -a], np.ones(len(k)))
    inside = np.nonzero(np.abs(climb - 1) > 0.02)[0]
    ks = inside[-1] + 1 if len(inside) else 0
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8))
    ax1.plot(np.arange(300), x[:300], color=GRAY, lw=0.8, label="raw")
    ax1.plot(np.arange(300), y[:300], color=ORANGE, lw=1.4, label="smoothed")
    ax1.set_xlabel("k")
    ax1.set_ylabel("altitude noise (m)")
    ax1.set_title(f"a = {a:.3f}: output noise {std:.3f} m (input 0.5 m)", fontsize=11)
    ax1.legend(loc="upper right", fontsize=9)
    ax2.plot(k, climb, ".-", color=BLUE, ms=4, lw=1)
    ax2.axhspan(0.98, 1.02, color=TEAL, alpha=0.15)
    ax2.axvline(ks, color=GRAY, ls=":", lw=1)
    ax2.set_xlabel("k")
    ax2.set_title(f"1 m climb: within 2% from k = {ks}", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_shaping(m=0, c=0.0):
    """H4: G(z) = 1.44/((z-0.2)(z+0.8)); command 1/(1+c) for k < m, then 1 (m = 0: plain step)."""
    n = 16
    k = np.arange(n)
    u = np.where(k < m, 1 / (1 + c), 1.0)
    y = signal.lfilter([0, 0, 1.44], [1, 0.6, -0.16], u)
    ok_u = np.all((u >= -1e-12) & (u <= 1 + 1e-12))
    ok_y = np.all(y <= 1.02) and np.all(np.abs(y[6:] - 1) <= 0.02)
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.step(k, u, where="post", color=GRAY, lw=1.4, label="command u(k)")
    ax.plot(k, y, "o-", color=ORANGE, ms=5, lw=1, label="head position y(k)")
    ax.axhspan(0.98, 1.02, color=BLUE, alpha=0.15, label="2% band")
    ax.axvline(6, color=GRAY, ls=":", lw=1)
    ax.set_ylim(-0.1, max(1.6, y.max() + 0.1, u.max() + 0.1))
    ax.set_xlabel("k")
    verdict = "PASS" if (ok_u and ok_y) else "FAIL"
    ax.set_title(f"m = {m}, c = {c:.3f}: max y {y.max():.3f}, max u {u.max():.3f}  ->  {verdict}",
                 fontsize=11)
    ax.legend(loc="lower right", fontsize=9)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, fixed
    interact(h1_two_lakes, release=FloatSlider(value=0.2, min=0.05, max=0.6, step=0.01,
                                               description="release", continuous_update=False))
    interact(h2_battery_pulse)
    interact(h3_smoothing, a=FloatSlider(value=0.5, min=0.0, max=0.99, step=0.002,
                                         readout_format=".3f", description="a",
                                         continuous_update=False))
    interact(h4_shaping, m=IntSlider(value=0, min=0, max=5, description="m",
                                     continuous_update=False),
             c=FloatSlider(value=0.0, min=-0.9, max=1.0, step=0.001, readout_format=".3f",
                           description="c", continuous_update=False))
except ImportError:
    h1_two_lakes()
    h2_battery_pulse()
    h3_smoothing()
    h4_shaping()